# ML - K-Vecinos más Cercanos (KNN)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import pickle

# Paso 1: Carga del conjunto de datos

In [2]:
df = pd.read_csv('../data/raw/winequality-red.csv', sep=';')
df['quality'].value_counts()

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

In [3]:
df['quality'] = np.select([df['quality'] <= 5, df['quality'] == 6, (df['quality'] > 6) & (df['quality'] <= 8)], [0, 1, 2])
df['quality'].value_counts()

quality
0    744
1    638
2    217
Name: count, dtype: int64

In [4]:
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,0
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,0
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,0
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,1
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,0
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,1
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,1
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,0


## Paso 2: Entrenamos el modelo KNN 

### Split (Segundo enfoque)

In [6]:
X = df.drop('quality', axis=1)
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
751,8.3,0.650,0.10,2.9,0.089,17.0,40.0,0.99803,3.29,0.55,9.5
370,6.9,0.765,0.02,2.3,0.063,35.0,63.0,0.99750,3.57,0.78,9.9
374,14.0,0.410,0.63,3.8,0.089,6.0,47.0,1.00140,3.01,0.81,10.8
537,8.1,0.825,0.24,2.1,0.084,5.0,13.0,0.99720,3.37,0.77,10.7
708,7.8,0.545,0.12,2.5,0.068,11.0,35.0,0.99600,3.34,0.61,11.6
...,...,...,...,...,...,...,...,...,...,...,...
368,10.3,0.340,0.52,2.8,0.159,15.0,75.0,0.99980,3.18,0.64,9.4
48,6.4,0.400,0.23,1.6,0.066,5.0,12.0,0.99580,3.34,0.56,9.2
772,9.5,0.570,0.27,2.3,0.082,23.0,144.0,0.99782,3.27,0.55,9.4
1231,7.8,0.815,0.01,2.6,0.074,48.0,90.0,0.99621,3.38,0.62,10.8


### Scaling - Escalado Mínimo-Máximo (variables numéricas)

In [7]:
min_max_scaler = MinMaxScaler()
# Ajustamos solo con los datos de entrenamiento
min_max_scaler.fit(X_train)

X_train_scal = min_max_scaler.transform(X_train)
X_train_scal = pd.DataFrame(X_train_scal, index=X_train.index, columns=X_train.columns)

X_test_scal = min_max_scaler.transform(X_test)
X_test_scal = pd.DataFrame(X_test_scal, index=X_test.index, columns=X_test.columns)

X_train_scal

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
751,0.321429,0.363014,0.10,0.137931,0.128548,0.225352,0.120141,0.584435,0.433071,0.131737,0.169231
370,0.196429,0.441781,0.02,0.096552,0.085142,0.478873,0.201413,0.545521,0.653543,0.269461,0.230769
374,0.830357,0.198630,0.63,0.200000,0.128548,0.070423,0.144876,0.831865,0.212598,0.287425,0.369231
537,0.303571,0.482877,0.24,0.082759,0.120200,0.056338,0.024735,0.523495,0.496063,0.263473,0.353846
708,0.276786,0.291096,0.12,0.110345,0.093489,0.140845,0.102473,0.435389,0.472441,0.167665,0.492308
...,...,...,...,...,...,...,...,...,...,...,...
368,0.500000,0.150685,0.52,0.131034,0.245409,0.197183,0.243816,0.714391,0.346457,0.185629,0.153846
48,0.151786,0.191781,0.23,0.048276,0.090150,0.056338,0.021201,0.420705,0.472441,0.137725,0.123077
772,0.428571,0.308219,0.27,0.096552,0.116861,0.309859,0.487633,0.569016,0.417323,0.131737,0.153846
1231,0.276786,0.476027,0.01,0.117241,0.103506,0.661972,0.296820,0.450808,0.503937,0.173653,0.369231


### Cargamos el modelo y lo entrenamos

In [8]:
with open('../models/08-knn-grid-model.pkl', 'rb') as f:
    knn_grid_model_2 = pickle.load(f)

In [9]:
knn_grid_model_2.fit(X_train_scal, y_train)

KNeighborsClassifier(metric='manhattan', n_neighbors=15, weights='distance')

### Predicciones

In [10]:
y_pred_test = knn_grid_model_2.predict(X_test_scal)

## Paso 3: Evaluamos el rendimiento

In [11]:
accuracy_score(y_test, y_pred_test)

0.728125

In [12]:
report_knn_grid_model_2 = classification_report(y_test, y_pred_test)
print(report_knn_grid_model_2)

              precision    recall  f1-score   support

           0       0.78      0.77      0.77       144
           1       0.68      0.68      0.68       131
           2       0.72      0.73      0.73        45

    accuracy                           0.73       320
   macro avg       0.72      0.73      0.73       320
weighted avg       0.73      0.73      0.73       320



In [23]:
#Guardamos el mejor modelo
with open('../models/08-knn-grid-model-2.pkl', 'wb') as f:
    pickle.dump(knn_grid_model_2, f)

> ## Conclusiones:
>
> - Hemos obtenido una mejora clara en el balance general de las clases, ahora tenemos mejor potencial de generalización
> - Mejoramos notablemente las métricas de la clase 0 (antes ignorada), ahora se predice bastante bien

## Paso 4: Probamos el modelo

### Creamos una muestra con los valores numéricos

In [28]:
nueva_muestra = pd.DataFrame([{'fixed acidity': 7.2,
                               'volatile acidity': 0.55,
                               'citric acid': 0.10,
                               'residual sugar': 2.0,
                               'chlorides': 0.08,
                               'free sulfur dioxide': 15.0,
                               'total sulfur dioxide': 50.0,
                               'density': 0.9965,
                               'pH': 3.30,
                               'sulphates': 0.65,
                               'alcohol': 10.0}])

nueva_muestra

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,7.2,0.55,0.1,2.0,0.08,15.0,50.0,0.9965,3.3,0.65,10.0


### Escalamos la muestra

In [17]:
nueva_muestra_scaled = min_max_scaler.transform(nueva_muestra)
nueva_muestra_scaled

array([[0.22321429, 0.29452055, 0.1       , 0.07586207, 0.11352254,
        0.1971831 , 0.15547703, 0.47209985, 0.44094488, 0.19161677,
        0.24615385]])

### Hacemos la predicción

In [30]:
pred_clase = knn_grid_model_2.predict(nueva_muestra_scaled)
f'predicion clase: {pred_clase[0]}'

/home/vscode/.local/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


'predicion clase: 1'

In [32]:
mapeo = {0: "El vino es de baja calidad (3-5) 🍷",
         1: "El vino es de calidad media (6) 🍷",
         2: "El vino es de alta calidad (7-8) 🍷"}

print("Predicción:", mapeo[pred_clase[0]])

Predicción: El vino es de calidad media (6) 🍷
